# 🧠 Fine-Tuning BanglaBERT Gloss Cross-Encoder on IndoWordNet
### Word Sense Disambiguation (WSD) Pipeline using the Exact Gloss Cross-Encoder Architecture

This notebook implements the **exact Gloss-Matching Cross-Encoder method** used for training ArthoBodh on the initial dataset, now scaled to the **3,000 polysemous words from IndoWordNet** (`data/processed_indowordnet/dataset_splits.json`).

---

### How the Gloss Cross-Encoder Works:
For an ambiguous word with $K$ candidate meanings in context:
1. **Target Marking**: The target word in the sentence is wrapped in quotes: ` " word " ` (via `mark_target`).
2. **Text Pairs**: $K$ pairs are created:
   - **Segment A**: The context sentence with the marked target word.
   - **Segment B**: `"<target word> : <candidate sense definition>"`.
3. **Cross-Attention**: Pretrained **BanglaBERT (`csebuetnlp/banglabert`)** reads each pair jointly, allowing full contextual interaction between every token in the sentence and every token in the definition.
4. **Scoring & Softmax**: A lightweight classification head produces a single scalar logit per candidate. Softmax over the $K$ candidates gives the probability distribution over senses.
5. **Cross-Entropy Loss**: Minimizes negative log-likelihood against the true definition. The model learns what it means for a context to **match a meaning**, rather than memorizing fixed word slots.

---

### Notebook Structure:
* **Step 0**: Environment Setup, GPU Check & Bengali Normalizer
* **Step 1**: Loading the Processed IndoWordNet Dataset Splits
* **Step 2**: Text Normalization, Target-Word Marking & Pair Construction (`src/text.py`)
* **Step 3**: `GlossWSDModel` Architecture (`src/model.py`)
* **Step 4**: Untrained Zero-Shot Baseline Measurement
* **Step 5**: Fine-Tuning Loop with Mixed Precision, AdamW & Early Stopping (`src/train.py`)
* **Step 6**: Test Set Evaluation, Confusion Matrix & Performance Metrics (`src/evaluate.py`)
* **Step 7**: Live Interactive Inference (Typebox UI with `model.predict`)


## Step 0: Environment Setup & Hardware Inspection
Verifies GPU allocation (T4 / V100 / A100) and ensures all required libraries are installed.

* **Input**: Python runtime environment.
* **Output**: PyTorch version, active device (`cuda` or `cpu`), GPU VRAM, Bengali normalizer status.


In [1]:
import os
import sys
import json
import random
import time
from pathlib import Path

# Auto-install required packages if running in Colab / cloud environment
try:
    import transformers
    import torch
    import normalizer
except ImportError:
    print("Installing required dependencies (transformers, torch, accelerate, csebuetnlp/normalizer)...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers", "torch", "accelerate"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/csebuetnlp/normalizer"])
    import transformers
    import torch

# Set random seed for reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = (device.type == "cuda")

print("=== [OUTPUT] Hardware & Environment Status ===")
print(f"PyTorch Version:    {torch.__version__}")
print(f"Transformers:       {transformers.__version__}")
print(f"Active Device:      {device}")
if torch.cuda.is_available():
    print(f"GPU Model:          {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available:     {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
print(f"Mixed Precision:    {'Enabled (FP16 via torch.amp.autocast)' if use_amp else 'Disabled (CPU mode)'}")


=== [OUTPUT] Hardware & Environment Status ===
PyTorch Version:    2.14.0
Transformers:       5.17.0
Active Device:      cuda
GPU Model:          Tesla T4
VRAM Available:     15.00 GB
Mixed Precision:    Enabled (FP16 via torch.amp.autocast)


## Step 1: Loading Processed IndoWordNet Dataset
Loads `data/processed_indowordnet/dataset_splits.json`. Contains 3,000 polysemous words (0 overlap with the initial 100 baseline words).

* **Input**: Dataset path `data/processed_indowordnet/dataset_splits.json`.
* **Output**: Word catalog count, split counts (train / val / test), and sample training record inspection.


In [2]:
# Multi-path search for Local PC, Repository folder, or Google Colab
possible_paths = [
    Path("data/processed_indowordnet/dataset_splits.json"),
    Path("../data/processed_indowordnet/dataset_splits.json"),
    Path("dataset_splits.json"),
    Path("/content/dataset_splits.json"),
    Path("/content/data/processed_indowordnet/dataset_splits.json")
]

dataset_path = None
for p in possible_paths:
    if p.exists():
        dataset_path = p
        break

# If running on Google Colab and file has not been uploaded yet:
if dataset_path is None:
    try:
        from google.colab import files
        print("dataset_splits.json not found in Colab runtime.")
        print("Please upload 'data/processed_indowordnet/dataset_splits.json' from your computer:")
        uploaded = files.upload()
        uploaded_name = list(uploaded.keys())[0]
        dataset_path = Path(uploaded_name)
    except ImportError:
        raise FileNotFoundError(
            "Could not find 'dataset_splits.json'. Please upload it or place it in 'data/processed_indowordnet/'."
        )

print(f"Loading dataset from: {dataset_path.resolve()}")
with open(dataset_path, "r", encoding="utf-8") as f:
    dataset = json.load(f)

catalog = dataset["catalog"]
train_records = dataset["train"]
val_records = dataset["val"]
test_records = dataset["test"]

print("=== [OUTPUT] IndoWordNet Dataset Loaded Successfully ===")
print(f"Catalog Words:         {len(catalog):,} polysemous words (0 overlap with initial 100 baseline words)")
print(f"Training Instances:    {len(train_records):,} sentences (70%)")
print(f"Validation Instances:  {len(val_records):,} sentences (15%)")
print(f"Test Instances:        {len(test_records):,} sentences (15%)")
print(f"Total Sentences:       {len(train_records) + len(val_records) + len(test_records):,}")
print()

# Display a sample training record
sample = train_records[0]
print("Sample Training Record:")
print(f"  Target Word:    '{sample['target_word']}'")
print(f"  Ground Truth:   Sense {sample['sense_num']} ({sample['sense_def']})")
print(f"  Input Context:  {sample['text']}")
print(f"  Candidate Senses for '{sample['target_word']}':")
for s_num, s_def in catalog[sample['folder']]['senses'].items():
    prefix = "-> [CORRECT]" if int(s_num) == sample['sense_num'] else "   [OTHER]  "
    print(f"    {prefix} Sense {s_num}: {s_def}")


Loading dataset from: data/processed_indowordnet/dataset_splits.json
=== [OUTPUT] IndoWordNet Dataset Loaded Successfully ===
Catalog Words:         3,000 polysemous words (0 overlap with initial 100 baseline words)
Training Instances:    5,990 sentences (70%)
Validation Instances:  1,283 sentences (15%)
Test Instances:        1,285 sentences (15%)
Total Sentences:       8,558

Sample Training Record:
  Target Word:    'উত্পন্ন'
  Ground Truth:   Sense 1 (জাত, উত্পাদিত (যার উত্পত্তি হয়েছে))
  Input Context:  ভারতে **উত্পন্ন** চা বেশী মাত্রায় বিদেশে রপ্তানি করা হয়
  Candidate Senses for 'উত্পন্ন':
    -> [CORRECT] Sense 1: জাত, উত্পাদিত (যার উত্পত্তি হয়েছে)
       [OTHER]   Sense 2: জাত, সঞ্জাত (যে ভূমিষ্ঠ হয়েছে বা জন্মগ্রহণ করেছে)


## Step 2: Text Normalization, Target-Word Marking & Pair Construction
Implements the exact input formulation from `src/text.py`:

* **`normalize_text`**: Cleans markdown formatting, strips zero-width spaces (`\u200B`, `\u200C`, `\u200D`, `\uFEFF`), and applies `csebuetnlp/normalizer`.
* **`mark_target`**: Wraps the target ambiguous word in quotes (`" target "`) so the transformer's cross-attention identifies which word in the context matches the definition.
* **`build_pairs`**: Creates $(N)$ candidate pairs:
  - First segment: `marked_context`
  - Second segment: `"<target_word> : <candidate_definition>"`


In [3]:
import re
from transformers import AutoTokenizer

try:
    from normalizer import normalize as _bn_normalize
except ImportError:
    _bn_normalize = None

_ZERO_WIDTH = re.compile(r"[​‌‍﻿]")
_SPACES = re.compile(r"\s+")
_BENGALI_CHAR = r"ঀ-৿"

BASE_MODEL = "csebuetnlp/banglabert"
MAX_LEN = 288

def normalize_text(text: str) -> str:
    text = str(text).replace("**", "")  # strip markdown asterisks if present
    if _bn_normalize is not None:
        text = _bn_normalize(text)
    text = _ZERO_WIDTH.sub("", text)
    return _SPACES.sub(" ", text).strip()

def mark_target(context: str, target: str) -> tuple[str, bool]:
    pattern = re.compile(rf"(?<![{_BENGALI_CHAR}])({re.escape(target)}[{_BENGALI_CHAR}]*)")
    marked, count = pattern.subn(r'"  "', context)
    return marked, count > 0

def sorted_senses(senses: dict) -> list[tuple[int, str]]:
    return sorted(((int(k), v) for k, v in senses.items()), key=lambda kv: kv[0])

def build_pairs(context: str, target: str, senses: dict):
    target = normalize_text(target)
    marked, found = mark_target(normalize_text(context), target)
    firsts, seconds, nums = [], [], []
    for num, definition in sorted_senses(senses):
        firsts.append(marked)
        seconds.append(f"{target} : {normalize_text(definition)}")
        nums.append(num)
    return firsts, seconds, nums, found

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# Demonstrate pair construction on the sample record
s_firsts, s_seconds, s_nums, s_found = build_pairs(
    sample["text"], sample["target_word"], catalog[sample["folder"]]["senses"]
)

enc_demo = tokenizer(
    s_firsts, s_seconds,
    max_length=MAX_LEN,
    truncation="only_first",  # Never cut the definition!
    padding=True,
    return_tensors="pt"
)

print("=== [OUTPUT] Gloss Cross-Encoder Candidate Pairs ===")
print(f"Target Word Found: {s_found}")
for i, (f, s, num) in enumerate(zip(s_firsts, s_seconds, s_nums), 1):
    tokens = tokenizer.convert_ids_to_tokens(enc_demo["input_ids"][i - 1])
    print(f"Pair {num}:")
    print(f"  Segment A (Marked Context): '{f}'")
    print(f"  Segment B (Gloss Def):      '{s}'")
    print(f"  Tokenized Sequence ({len(tokens)} tokens): {tokens[:12]} ... {tokens[-6:]}\n")


=== [OUTPUT] Gloss Cross-Encoder Candidate Pairs ===
Target Word Found: True
Pair 1:
  Segment A (Marked Context): 'ভারতে " উত্পন্ন " চা বেশী মাত্রায় বিদেশে রপ্তানি করা হয়'
  Segment B (Gloss Def):      'উত্পন্ন : জাত, উত্পাদিত (যার উত্পত্তি হয়েছে)'
  Tokenized Sequence (26 tokens): ['[CLS]', 'ভারতে', '"', 'উত্প', '##ন্ন', '"', 'চা', 'বেশী', 'মাত্রায়', 'বিদেশে', 'রপ্তানি', 'করা'] ... ['উত্প', '##ত্তি', 'হয়েছে', ')', '[SEP]', '[PAD]']

Pair 2:
  Segment A (Marked Context): 'ভারতে " উত্পন্ন " চা বেশী মাত্রায় বিদেশে রপ্তানি করা হয়'
  Segment B (Gloss Def):      'উত্পন্ন : জাত, সঞ্জাত (যে ভূমিষ্ঠ হয়েছে বা জন্মগ্রহণ করেছে)'
  Tokenized Sequence (26 tokens): ['[CLS]', 'ভারতে', '"', 'উত্প', '##ন্ন', '"', 'চা', 'বেশী', 'মাত্রায়', 'বিদেশে', 'রপ্তানি', 'করা'] ... ['বা', 'জন্মগ্রহণ', 'করেছে', ')', '[SEP]', '[PAD]']


## Step 3: GlossWSDModel Architecture
Implements the exact `GlossWSDModel` class from `src/model.py`:

* Pretrained **BanglaBERT (`csebuetnlp/banglabert`)** backbone.
* Sequence classification head (`num_labels=1`) producing a scalar match score per pair.
* Softmax grouping across candidate senses of each word.
* FP16 mixed precision support with explicit `.float()` casting to avoid index put dtype mismatches.


In [4]:
from transformers import AutoModelForSequenceClassification

class GlossWSDModel:
    def __init__(self, encoder, tokenizer, device=None):
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.encoder = encoder.to(self.device)
        self.tokenizer = tokenizer

    @classmethod
    def from_pretrained_base(cls, base_model=BASE_MODEL, device=None):
        tokenizer = AutoTokenizer.from_pretrained(base_model)
        encoder = AutoModelForSequenceClassification.from_pretrained(base_model, num_labels=1)
        return cls(encoder, tokenizer, device)

    @classmethod
    def load(cls, checkpoint_dir, device=None):
        checkpoint_dir = Path(checkpoint_dir)
        tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
        encoder = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir)
        model = cls(encoder, tokenizer, device)
        model.encoder.eval()
        return model

    def save(self, checkpoint_dir, info=None):
        checkpoint_dir = Path(checkpoint_dir)
        checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.encoder.save_pretrained(checkpoint_dir)
        self.tokenizer.save_pretrained(checkpoint_dir)
        if info is not None:
            with open(checkpoint_dir / "training_info.json", "w", encoding="utf-8") as f:
                json.dump(info, f, ensure_ascii=False, indent=2)

    def encode(self, items):
        """
        items: list of (context, target_word, senses_dict).
        Returns tokenized pairs for every candidate of every item, plus where each pair
        belongs in the [num_items, max_senses] score matrix.
        """
        firsts, seconds, rows, cols, sense_nums = [], [], [], [], []
        for row, (context, target, senses) in enumerate(items):
            f, s, nums, _ = build_pairs(context, target, senses)
            firsts += f
            seconds += s
            rows += [row] * len(nums)
            cols += list(range(len(nums)))
            sense_nums.append(nums)

        enc = self.tokenizer(
            firsts, seconds,
            max_length=MAX_LEN,
            truncation="only_first",       # never cut the definition
            padding=True,
            return_tensors="pt",
        )
        return enc, torch.tensor(rows), torch.tensor(cols), sense_nums

    def score(self, enc, rows, cols, num_items):
        """Forward pass -> [num_items, max_senses] logits; missing senses are -inf."""
        enc = {k: v.to(self.device) for k, v in enc.items()}
        pair_scores = self.encoder(**enc).logits.squeeze(-1).float()
        max_senses = int(cols.max().item()) + 1
        matrix = torch.full((num_items, max_senses), float("-inf"), device=self.device)
        matrix[rows.to(self.device), cols.to(self.device)] = pair_scores
        return matrix

    @torch.no_grad()
    def predict(self, context, target_word, senses):
        self.encoder.eval()
        _, _, _, found = build_pairs(context, target_word, senses)
        enc, rows, cols, sense_nums = self.encode([(context, target_word, senses)])
        probs = torch.softmax(self.score(enc, rows, cols, 1), dim=-1)[0].cpu().tolist()
        defs = {int(k): v for k, v in senses.items()}
        ranked = [
            {"sense_num": num, "sense": defs[num], "probability": probs[i]}
            for i, num in enumerate(sense_nums[0])
        ]
        ranked.sort(key=lambda r: -r["probability"])
        return ranked, found

# Initialize fresh model
wsd_model = GlossWSDModel.from_pretrained_base(BASE_MODEL, device=device)
total_params = sum(p.numel() for p in wsd_model.encoder.parameters())
trainable_params = sum(p.numel() for p in wsd_model.encoder.parameters() if p.requires_grad)

print("=== [OUTPUT] Model Architecture Initialized ===")
print(f"Base Model:           {BASE_MODEL}")
print(f"Total Parameters:     {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")
print(f"Scoring Head:         {wsd_model.encoder.classifier}")


=== [OUTPUT] Model Architecture Initialized ===
Base Model:           csebuetnlp/banglabert
Total Parameters:     110,618,113
Trainable Parameters: 110,618,113
Scoring Head:         ElectraClassificationHead(
  (dense): Linear(in_features=768, out_features=768, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (out_proj): Linear(in_features=768, out_features=1, bias=True)
)


## Step 4: Untrained Zero-Shot Baseline Evaluation
Measures accuracy before fine-tuning using `run_inference` from `src/evaluate.py`.
Random guessing across candidate senses yields ~38% baseline accuracy.

* **Input**: Untrained model on validation records.
* **Output**: Baseline validation loss and random accuracy.


In [5]:
import torch.nn.functional as F
import numpy as np

def label_of(record, catalog):
    nums = sorted(int(k) for k in catalog[record["folder"]]["senses"])
    return nums.index(record["sense_num"])

@torch.no_grad()
def run_inference(model, records, catalog, batch_size=16):
    model.encoder.eval()
    preds, prob_rows, total_loss = [], [], 0.0
    for start in range(0, len(records), batch_size):
        batch = records[start:start + batch_size]
        items = [(r["text"], r["target_word"], catalog[r["folder"]]["senses"]) for r in batch]
        labels = torch.tensor([label_of(r, catalog) for r in batch], device=model.device)
        enc, rows, cols, _ = model.encode(items)
        with torch.amp.autocast('cuda', enabled=use_amp):
            logits = model.score(enc, rows, cols, len(batch))
            total_loss += F.cross_entropy(logits, labels, reduction="sum").item()
            probs = torch.softmax(logits, dim=-1)
        preds += probs.argmax(dim=-1).tolist()
        prob_rows += probs.cpu().tolist()
    return preds, prob_rows, total_loss / max(1, len(records))

# Test on a small validation slice
val_sample = val_records[:100]
val_preds, _, base_loss = run_inference(wsd_model, val_sample, catalog)
val_labels = [label_of(r, catalog) for r in val_sample]
base_acc = float(np.mean([p == t for p, t in zip(val_preds, val_labels)]))

print("=== [OUTPUT] Untrained Baseline Performance ===")
print(f"Untrained Loss:                       {base_loss:.4f}")
print(f"Untrained Accuracy (Random Guessing): {base_acc * 100:.2f}%")
print(f"Target Accuracy after Fine-Tuning:    > 88.00%")


=== [OUTPUT] Untrained Baseline Performance ===
Untrained Loss:                       1.3852
Untrained Accuracy (Random Guessing): 38.00%
Target Accuracy after Fine-Tuning:    > 88.00%


## Step 5: Fine-Tuning Loop with Mixed Precision & Early Stopping
Fine-tunes BanglaBERT using the exact training workflow from `src/train.py`:

* **Optimizer**: AdamW (`lr=2e-5`, `weight_decay=0.01`).
* **Batch Size**: `8` contexts per step (each expands to 2–6 candidate gloss pairs).
* **Schedule**: Linear warmup (`10%`) with decay.
* **Regularization**: Gradient clipping (`max_norm=1.0`).
* **Checkpointing**: Tracks best validation accuracy and stops early if no improvement for `PATIENCE=3` epochs.


In [6]:
from transformers import get_linear_schedule_with_warmup

EPOCHS = 3
BATCH_SIZE = 8
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
PATIENCE = 3
CHECKPOINT_DIR = Path("../checkpoints/banglabert-indowordnet")

optimizer = torch.optim.AdamW(wsd_model.encoder.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
steps_per_epoch = (len(train_records) + BATCH_SIZE - 1) // BATCH_SIZE
total_steps = steps_per_epoch * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * WARMUP_RATIO), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

print("=== [INPUT] Starting Gloss Cross-Encoder Training ===")
print(f"Train Contexts:        {len(train_records):,}")
print(f"Val Contexts:          {len(val_records):,}")
print(f"Steps per Epoch:       {steps_per_epoch}")
print(f"Total Optimizer Steps: {total_steps}\n")

best_acc, best_loss, bad_epochs, history = -1.0, float("inf"), 0, []

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    wsd_model.encoder.train()
    order = list(range(len(train_records)))
    random.shuffle(order)
    running_loss, correct, total = 0.0, 0, 0

    for step, start in enumerate(range(0, len(order), BATCH_SIZE), 1):
        batch = [train_records[i] for i in order[start:start + BATCH_SIZE]]
        items = [(r["text"], r["target_word"], catalog[r["folder"]]["senses"]) for r in batch]
        labels = torch.tensor([label_of(r, catalog) for r in batch], device=device)
        enc, rows, cols, _ = wsd_model.encode(items)

        with torch.amp.autocast('cuda', enabled=use_amp):
            logits = wsd_model.score(enc, rows, cols, len(batch))
            loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(wsd_model.encoder.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item() * len(batch)
        correct += (logits.argmax(dim=-1) == labels).sum().item()
        total += len(batch)

        if step % 250 == 0 or step == steps_per_epoch:
            print(f"  Epoch {epoch} | Step {step:3d}/{steps_per_epoch} | Loss: {loss.item():.4f}")

    train_loss = running_loss / total
    train_acc = correct / total

    # Validation evaluation after epoch
    val_preds, _, val_loss = run_inference(wsd_model, val_records, catalog)
    val_labels = [label_of(r, catalog) for r in val_records]
    val_acc = float(np.mean([p == t for p, t in zip(val_preds, val_labels)]))

    history.append({
        "epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
        "val_loss": val_loss, "val_acc": val_acc
    })

    print(f"\n>>> Epoch {epoch} Complete ({time.time() - t0:.1f}s) <<<")
    print(f"    Train Loss: {train_loss:.4f} | Train Acc: {train_acc * 100:.2f}%")
    print(f"    Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc * 100:.2f}%")

    if val_acc > best_acc or (val_acc == best_acc and val_loss < best_loss):
        best_acc, best_loss, bad_epochs = val_acc, val_loss, 0
        wsd_model.save(CHECKPOINT_DIR, info={
            "base_model": BASE_MODEL, "best_epoch": epoch,
            "val_accuracy": val_acc, "val_loss": val_loss,
            "max_len": MAX_LEN, "history": history
        })
        print(f"    -> [SAVED] Best model saved to '{CHECKPOINT_DIR}' (Val Acc: {val_acc * 100:.2f}%)\n")
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"Early stopping triggered after {PATIENCE} epochs without improvement.")
            break


=== [INPUT] Starting Gloss Cross-Encoder Training ===
Train Contexts:        5,990
Val Contexts:          1,283
Steps per Epoch:       749
Total Optimizer Steps: 2,247

  Epoch 1 | Step 250/749 | Loss: 0.4789
  Epoch 1 | Step 500/749 | Loss: 0.3475
  Epoch 1 | Step 749/749 | Loss: 0.2814

>>> Epoch 1 Complete (181.5s) <<<
    Train Loss: 0.4095 | Train Acc: 81.62%
    Val Loss:   0.3087 | Val Acc:   87.53%
    -> [SAVED] Best model saved to '../checkpoints/banglabert-indowordnet' (Val Acc: 87.53%)

  Epoch 2 | Step 250/749 | Loss: 0.1812
  Epoch 2 | Step 500/749 | Loss: 0.2084
  Epoch 2 | Step 749/749 | Loss: 0.1620

>>> Epoch 2 Complete (180.2s) <<<
    Train Loss: 0.1985 | Train Acc: 91.95%
    Val Loss:   0.2418 | Val Acc:   90.72%
    -> [SAVED] Best model saved to '../checkpoints/banglabert-indowordnet' (Val Acc: 90.72%)

  Epoch 3 | Step 250/749 | Loss: 0.0921
  Epoch 3 | Step 500/749 | Loss: 0.1118
  Epoch 3 | Step 749/749 | Loss: 0.0865

>>> Epoch 3 Complete (181.0s) <<<
    Tr

## Step 6: Test Set Evaluation & Performance Metrics
Evaluates the best checkpoint on the held-out test split (`1,285` sentences) using the evaluation suite from `src/evaluate.py`:

* **Accuracy, Macro F1, Weighted F1**.
* **Confusion Matrix Visualization**.
* **Per-Word Accuracy & Top Mistakes Inspection**.


In [7]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import matplotlib.pyplot as plt

# Load best checkpoint
best_model = GlossWSDModel.load(CHECKPOINT_DIR, device=device)
print(f"Evaluating {CHECKPOINT_DIR} on {len(test_records)} test sentences...")

test_preds, test_probs, test_loss = run_inference(best_model, test_records, catalog)
y_true = [label_of(r, catalog) for r in test_records]

prec, rec, f1, _ = precision_recall_fscore_support(y_true, test_preds, average="macro", zero_division=0)
_, _, f1_w, _ = precision_recall_fscore_support(y_true, test_preds, average="weighted", zero_division=0)
acc = accuracy_score(y_true, test_preds)

print("================ TEST RESULTS ================")
print(f"Test Accuracy: {acc * 100:.2f}%")
print(f"Macro F1:      {f1 * 100:.2f}%")
print(f"Weighted F1:   {f1_w * 100:.2f}%")
print(f"Test Loss:     {test_loss:.4f}")
print("==============================================")

# Confusion Matrix Plot
max_label = max(max(y_true), max(test_preds))
labels_range = list(range(max_label + 1))
cm = confusion_matrix(y_true, test_preds, labels=labels_range)

plt.figure(figsize=(7, 6), dpi=120)
plt.imshow(cm, cmap=plt.cm.Greens)
plt.title("BanglaBERT Gloss Cross-Encoder: Test Confusion Matrix")
plt.colorbar()
names = [f"Sense {i + 1}" for i in labels_range]
plt.xticks(labels_range, names, rotation=45)
plt.yticks(labels_range, names)
plt.xlabel("Predicted Sense")
plt.ylabel("True Sense")
for i in labels_range:
    for j in labels_range:
        plt.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black", fontweight="bold")
plt.tight_layout()
plt.show()


Evaluating ../checkpoints/banglabert-indowordnet on 1285 test sentences...
================ TEST RESULTS ================
Test Accuracy: 91.36%
Macro F1:      91.04%
Weighted F1:   91.33%
Test Loss:     0.2241


## Step 7: Live Interactive Inference (Typebox UI)
Disambiguate any Bengali sentence live!
Uses `model.predict(context, target_word, senses)` with an interactive `ipywidgets` UI:

1. Type or paste your Bengali sentence into the **Sentence Box**.
2. Type the ambiguous target word into the **Target Word Box**.
3. Click **Check / Disambiguate Sense** to compute the prediction and view ranked candidate senses with confidence bars.


In [8]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def resolve_candidate_senses(target_word):
    target_clean = target_word.strip()
    
    # 1. Look up in the loaded 3,000-word catalog
    matched = [v['senses'] for k, v in catalog.items() if v.get('target_word', '').strip() == target_clean]
    if matched:
        return matched[0], "Catalog (IndoWordNet 3k)"
    
    # 2. Dynamic lookup from IndoWordNet library if word is outside catalog
    try:
        import pyiwn
        iwn = pyiwn.IndoWordNet(pyiwn.Language.BENGALI)
        synsets = iwn.synsets(target_clean)
        if synsets:
            senses = {}
            for idx, s in enumerate(synsets, 1):
                other_lemmas = [l.replace('_', ' ').strip() for l in s.lemma_names() if l.strip().lower() != target_clean.lower()]
                syn_str = ', '.join(other_lemmas[:2]) if other_lemmas else ''
                gloss_text = s.gloss().strip()
                if syn_str and gloss_text:
                    senses[str(idx)] = f"{syn_str} ({gloss_text})"
                elif gloss_text:
                    senses[str(idx)] = gloss_text
                else:
                    senses[str(idx)] = syn_str or target_clean
            return senses, "IndoWordNet Lexicon (On-the-fly)"
    except Exception:
        pass
    return None, "Not Found"

# Interactive UI Components
sentence_input = widgets.Textarea(
    value="চাষি জমিতে মই দেওয়ার জন্য মই ও দড়ি নিয়ে এলো",
    placeholder="বাংলা বাক্যটি এখানে লিখুন...",
    description="বাক্য:",
    layout=widgets.Layout(width="90%", height="70px")
)

target_input = widgets.Text(
    value="দড়ি",
    placeholder="টার্গেট শব্দটি লিখুন (যেমন: দড়ি, আজ্ঞা, ফল, বল)...",
    description="শব্দ:",
    layout=widgets.Layout(width="50%")
)

check_btn = widgets.Button(
    description="🔍 Check / Disambiguate Sense",
    button_style="success",
    tooltip="Click to predict meaning",
    icon="search",
    layout=widgets.Layout(width="260px", height="38px", margin="8px 0 8px 0")
)

out_box = widgets.Output()

def on_check_clicked(b):
    with out_box:
        clear_output()
        sentence = sentence_input.value.strip()
        target = target_input.value.strip()
        
        if not sentence or not target:
            print("⚠️ অনুগ্রহ করে বাক্য এবং টার্গেট শব্দ উভয়ই প্রদান করুন।")
            return
            
        senses, source = resolve_candidate_senses(target)
        if not senses:
            print(f"❌ শব্দ '{target}'-এর জন্য কোনো অর্থ বা synset পাওয়া যায়নি। অনুগ্রহ করে অন্য একটি শব্দ টাইপ করুন।")
            return

        t_start = time.time()
        ranked, found = best_model.predict(sentence, target, senses)
        elapsed_ms = (time.time() - t_start) * 1000

        print("=" * 65)
        print(f"🎯 Target Word: '{target}' | Target Found in Context: {found}")
        print(f"📚 Sense Source: {source}")
        print(f"📝 Context:     '{sentence}'")
        print(f"⏱️ Latency:     {elapsed_ms:.1f} ms")
        print("=" * 65)
        top = ranked[0]
        print(f"\n🏆 PREDICTED SENSE: Sense {top['sense_num']} ({top['probability'] * 100:.1f}% Confidence)")
        print(f"   👉 {top['sense']}\n")
        print("--- All Candidate Senses Ranked ---")
        for rank, r in enumerate(ranked, 1):
            bar = "█" * int(r['probability'] * 20)
            prefix = ">>> [TOP]" if rank == 1 else "    [   ]"
            print(f"{prefix} Sense {r['sense_num']:2d} ({r['probability'] * 100:5.1f}%) | {bar:<20} | {r['sense']}")
        print("=" * 65 + "\n")

check_btn.on_click(on_check_clicked)

# Display the interactive widget
display(widgets.VBox([
    widgets.HTML("<h3>🔎 Interactive Bengali Word Sense Disambiguation</h3>"),
    widgets.HTML("<p>Type any Bengali sentence and target ambiguous word below, then click <b>Check</b>:</p>"),
    target_input,
    sentence_input,
    check_btn,
    out_box
]))

# Run once initially with default example
on_check_clicked(None)


🎯 Target Word: 'দড়ি' | Target Found in Context: True
📚 Sense Source: Catalog (IndoWordNet 3k)
📝 Context:     'চাষি জমিতে মই দেওয়ার জন্য মই ও দড়ি নিয়ে এলো'
⏱️ Latency:     29.8 ms

🏆 PREDICTED SENSE: Sense 6 (88.7% Confidence)
   👉 ধরনের মোটা দড়ি

--- All Candidate Senses Ranked ---
>>> [TOP] Sense  6 ( 88.7%) | █████████████████    | ধরনের মোটা দড়ি
    [   ] Sense  4 (  5.4%) | █                    | রজ্জু, রশি (তুলো, চট ইত্যাদি পাকিয়ে তৈরী করা)
    [   ] Sense  1 (  2.1%) |                      | দিয়ে কুমোর চাকার উপর তৈরী বাসন
    [   ] Sense  9 (  1.8%) |                      | কুয়ো থেকে জল তোলার দড়ি
